# 5.2 风险平价策略 (Risk Parity)

## 学习目标
- 理解风险平价的核心思想：让每只资产对组合风险的贡献相等
- 用数值优化求解风险平价权重
- 对比 Markowitz、等权、风险平价三种方法

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize

plt.rcParams['figure.figsize'] = (12, 5)

# 多元资产池
tickers = ['SPY', 'TLT', 'GLD', 'QQQ', 'EFA']
labels = ['美股(SPY)', '长债(TLT)', '黄金(GLD)', '纳指(QQQ)', '国际股(EFA)']

prices = yf.download(tickers, start='2015-01-01', end='2024-01-01',
                     progress=False)['Close'].dropna()
returns = prices.pct_change().dropna()

mu = returns.mean() * 252
cov = returns.cov() * 252
n = len(tickers)
print('数据加载完成 ✅')
print('年化收益率:')
print(mu.round(3))

## 1. 风险贡献分析

对于给定权重 $\mathbf{w}$，第 $i$ 个资产的**边际风险贡献（MRC）**为：

$$MRC_i = \frac{(\mathbf{\Sigma w})_i}{\sigma_p}$$

第 $i$ 个资产的**总风险贡献（TRC）**：

$$TRC_i = w_i \times MRC_i$$

风险平价的目标：让所有 $TRC_i$ 相等。

In [ ]:
def portfolio_vol(w, cov):
    return np.sqrt(w @ cov @ w)

def risk_contributions(w, cov):
    """返回各资产的风险贡献（归一化，总和=1）"""
    w = np.array(w)
    sigma = portfolio_vol(w, cov)
    mrc = cov @ w / sigma
    trc = w * mrc
    return trc / trc.sum()  # 归一化

# 等权组合的风险贡献
equal_w = np.ones(n) / n
eq_rc = risk_contributions(equal_w, cov)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, eq_rc, color='steelblue', alpha=0.8)
ax.axhline(1 / n, color='red', linestyle='--', label=f'平均 = {1/n:.0%}')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_title('等权组合各资产风险贡献（并不相等！）', fontsize=12)
ax.set_ylabel('风险贡献占比')
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
print('观察：风险并没有被均等分配！股票往往占据大部分风险。')

## 2. 求解风险平价权重

In [ ]:
def risk_parity_objective(w, cov):
    """目标：最小化所有资产风险贡献的方差（使其尽量相等）"""
    w = np.array(w)
    sigma = portfolio_vol(w, cov)
    mrc = cov @ w / sigma
    trc = w * mrc  # 各资产总风险贡献（绝对值）
    target = sigma / n   # 目标：每人贡献 1/n
    return np.sum((trc - target) ** 2)

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(0.01, 1.0)] * n  # 每个资产至少 1%
w0 = equal_w

result = minimize(risk_parity_objective, w0, args=(cov,),
                   method='SLSQP', bounds=bounds, constraints=constraints,
                   options={'ftol': 1e-12, 'maxiter': 1000})

rp_w = result.x
rp_rc = risk_contributions(rp_w, cov)

print('风险平价权重：')
for ticker, label, w, rc in zip(tickers, labels, rp_w, rp_rc):
    print(f'  {label:<12}: 权重={w:.2%}, 风险贡献={rc:.2%}')

## 3. 三种策略权重对比

In [ ]:
# 最大夏普比率组合（来自 Markowitz）
def neg_sharpe(w, mu, cov, rf=0.04):
    ret = w @ mu
    vol = portfolio_vol(w, cov)
    return -(ret - rf) / vol

res_sr = minimize(neg_sharpe, w0, args=(mu, cov),
                   method='SLSQP', bounds=[(0, 1)] * n,
                   constraints=constraints)
sr_w = res_sr.x

# 权重对比图
x = np.arange(n)
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, equal_w, width, label='等权', color='steelblue', alpha=0.8)
ax.bar(x,         rp_w,    width, label='风险平价', color='orange', alpha=0.8)
ax.bar(x + width, sr_w,    width, label='最大夏普', color='green', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_title('三种组合策略权重对比', fontsize=13)
ax.set_ylabel('权重')
ax.legend()
ax.grid(alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

## 4. 历史回测对比

In [ ]:
def backtest_portfolio(weights, ret_df, rebalance='ME'):
    """简单组合回测（定期再平衡）"""
    port_ret = []

    for period_start, group in ret_df.resample(rebalance):
        period_ret = (group * weights).sum(axis=1)
        port_ret.append(period_ret)

    return pd.concat(port_ret)

ret_eq = backtest_portfolio(equal_w, returns)
ret_rp = backtest_portfolio(rp_w, returns)
ret_sr = backtest_portfolio(sr_w, returns)

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

# 累积净值
for ret, label, color in [
    (ret_eq, '等权', 'steelblue'),
    (ret_rp, '风险平价', 'orange'),
    (ret_sr, '最大夏普', 'green')
]:
    cum = (1 + ret).cumprod()
    axes[0].plot(cum.index, cum.values, label=label, linewidth=1.8, color=color)

axes[0].set_title('三种组合策略累积净值对比', fontsize=13)
axes[0].legend()
axes[0].grid(alpha=0.3)

# 各组合的风险贡献
rc_data = pd.DataFrame({
    '等权': risk_contributions(equal_w, cov),
    '风险平价': rp_rc,
    '最大夏普': risk_contributions(sr_w, cov),
}, index=labels)

rc_data.plot(kind='bar', ax=axes[1], color=['steelblue', 'orange', 'green'], alpha=0.8)
axes[1].axhline(1/n, color='red', linestyle='--', alpha=0.5, label=f'目标平均 {1/n:.0%}')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].set_title('各组合风险贡献对比', fontsize=13)
axes[1].set_xticklabels(labels, rotation=15)
axes[1].legend()
axes[1].grid(alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

## 5. 策略绩效汇总

In [ ]:
def quick_metrics(ret, name):
    n_days = len(ret)
    total = (1 + ret).prod() - 1
    ann_ret = (1 + total) ** (252 / n_days) - 1
    ann_vol = ret.std() * np.sqrt(252)
    sharpe = (ann_ret - 0.04) / ann_vol
    cum = (1 + ret).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min()
    return pd.Series({
        '年化收益': f'{ann_ret:.2%}',
        '年化波动': f'{ann_vol:.2%}',
        '夏普比率': f'{sharpe:.2f}',
        '最大回撤': f'{mdd:.2%}'
    }, name=name)

pd.DataFrame([
    quick_metrics(ret_eq, '等权'),
    quick_metrics(ret_rp, '风险平价'),
    quick_metrics(ret_sr, '最大夏普'),
]).T

## 🎯 练习

1. 加入加密货币 ETF（如 `GBTC`），观察其极高波动率如何影响风险平价权重。
2. 实现**等风险贡献（ERC）+ 动量过滤**：只纳入过去3月正收益的资产。
3. 对比月频 vs 季频再平衡的结果差异。

---
**下一模块** → `../06_ml_trading/01_feature_selection.ipynb`